In [1]:
import math
import random
import time
from bisect import bisect_left

In [2]:
SEED = 42
N = 10_000
QUERY_COUNT = 100
DAILY_QUERIES = 5000

# ДЗ №1. Сложность и бинарный поиск

Курс «Алгоритмы и структуры данных» · по материалам лекции 1

**Фамилия, группа:** [REDACTED NAME], [REDACTED GROUP]

## Как работать с этим ноутбуком

- Демо-ячейки уже исполнены — выводы видны сразу. Полезно прогнать их самому:
  откройте ноутбук в Jupyter и идите сверху вниз, Shift+Enter.
- Ячейки с заголовком **«### Ответ»** — ваши: пишите прямо в них, заглушку затрите.
- В ячейках с пометкой `# <-- можно менять здесь` — меняйте числа и перезапускайте:
  ноутбук ваш, экспериментируйте.
- Нужен только Python 3.9 и новее. Никаких пакетов ставить не нужно, интернета тоже.
- Точные числа на вашей машине будут другими — это нормально и ожидаемо.
  Важен порядок величин, а не конкретные миллисекунды.

## Шпаргалка с лекции

**Алгоритм** — точная конечная последовательность шагов, которая получает вход
и выдаёт правильный выход для любого допустимого входа. Каждое свойство важно:
«посолить по вкусу» — уже не алгоритм, шаг не однозначен.

**Сложность** — скорость роста числа шагов, а не время. Константы выбрасываем:
O(n² + 100n) — это O(n²).

Лестница классов: O(1) < O(log n) < O(n) < O(n log n) < O(n²) < O(n³) < O(2ⁿ).

**Бинарный поиск**: массив отсортирован; каждый шаг делит диапазон пополам;
худший случай — около log₂ n шагов.

---

## Задача 1. Алгоритмы и не алгоритмы у вас дома

Найдите в быту — дома, в инструкциях к технике, в рецептах — **три алгоритма**
и **три не-алгоритма**. По каждому примеру одно-два предложения: какие свойства
алгоритма выполняются или нарушаются.

Для калибровки, пример (сами его не используйте):

> «Заваривать чай, пока не станет вкусно» — не алгоритм: шаг не однозначен,
> «вкусно» не определено, нет критерия остановки.

Заполните таблицу в ячейке ниже.

| Пример | Алгоритм? | Почему |
| --- | --- | --- |
| Из конечного списка покупок выписать в новый список все позиции с количеством больше нуля, просмотрев каждую ровно один раз. | Да | Вход и проверка определены, после последней позиции работа заканчивается; результат содержит ровно нужные позиции. |
| Для каждой из десяти пронумерованных коробок приклеить этикетку с её номером, начиная с первой и заканчивая десятой. | Да | Действия однозначны, число шагов конечно, результат проверяется по номерам. |
| Для заданного целого числа минут получить часы целочисленным делением на 60, а оставшиеся минуты — остатком от деления. | Да | Для любого неотрицательного целого входа определены две операции и точный результат. |
| Убирать комнату, пока она не станет уютной. | Нет | Критерий «уютно» не определён, поэтому неизвестно, когда завершать работу. |
| Выбрать из списка покупок что-нибудь подходящее. | Нет | Не задано правило выбора, разные исполнители могут получить разные результаты. |
| Каждую минуту проверять пустой почтовый ящик, пока не придёт письмо, даже если отправка письма не обещана. | Нет | В допустимой ситуации без письма процедура не завершится; конечность не гарантирована. |

---

## Задача 2. Сколько шагов у вложенных циклов

```python
for i in range(n):
    for j in range(n * n):
        print(i * j)
```

Оцените сложность этого фрагмента — и объясните рассуждение, не только ответ.
Здесь легко ошибиться, так что проверьте себя экспериментом в ячейке ниже:
она считает шаги вместо `print` и печатает, как растёт счётчик с ростом n.

In [3]:
def count_steps(n):
    steps = 0
    for i in range(n):
        for j in range(n * n):
            steps += 1  # вместо print(i * j)
    return steps


for n in [5, 10, 20, 40]:
    s = count_steps(n)
    print(f"n={n:3d}: шагов {s:7d}, шагов/n^2 = {s / n**2:8.1f}, шагов/n^3 = {s / n**3:.3f}")

n=  5: шагов     125, шагов/n^2 =      5.0, шагов/n^3 = 1.000
n= 10: шагов    1000, шагов/n^2 =     10.0, шагов/n^3 = 1.000
n= 20: шагов    8000, шагов/n^2 =     20.0, шагов/n^3 = 1.000
n= 40: шагов   64000, шагов/n^2 =     40.0, шагов/n^3 = 1.000


**Наблюдение.** В колонке «шагов/n²» получаются 5, 10, 20 и 40: отношение растёт вместе с n.
Колонка «шагов/n³» всегда равна 1, поэтому число шагов точно равно n³.

### Ответ (задача 2)

Внешний цикл делает n итераций, и на каждой внутренний делает n² итераций.
Итого n × n² = n³ выполнений тела: точный порядок роста Θ(n³), верхняя оценка O(n³).
При удвоении n число выполнений тела увеличивается в восемь раз.
Это оценка количества итераций; стоимость форматирования и вывода чисел отдельно не учитывается.

---

## Задача 3. Сортировать или искать как есть?

Вы — программист в небольшой компании. База из 10 000 записей о клиентах,
записи лежат в случайном порядке, и каждый день по ней делают около 5 000
поисков. Руководитель спрашивает: «Гнать линейно или один раз отсортировать
и искать бинарно?» Обоснуйте числом — своим, с вашей машины.

In [4]:
random.seed(SEED)

db = [random.randint(0, 1_000_000) for _ in range(N)]
queries = [random.randint(0, 1_000_000) for _ in range(QUERY_COUNT)]  # <-- можно менять здесь


def linear_search(data, target):
    for i, x in enumerate(data):
        if x == target:
            return i
    return -1


sorted_db = sorted(db)


def binary_search(data, target):
    i = bisect_left(data, target)
    return i if i < len(data) and data[i] == target else -1


print(f"записей в базе: {len(db):,}")
print(f"первые 5 записей: {db[:5]}")

записей в базе: 10,000
первые 5 записей: [670487, 116739, 26225, 777572, 288389]


**Прогноз до запуска:** при 5000 запросах к неизменной базе предварительная сортировка с бинарным поиском
даст выигрыш как минимум в 100 раз. Это проверяемая гипотеза, а не заранее известный результат замера.

In [5]:
# Замер: 100 поисков каждого вида, дальше пересчёт на день (5 000 поисков).
# Запуск займёт пару секунд.

t0 = time.perf_counter()
for q in queries:
    linear_search(db, q)
t_linear_100 = time.perf_counter() - t0

t0 = time.perf_counter()
for q in queries:
    binary_search(sorted_db, q)
t_binary_100 = time.perf_counter() - t0

t0 = time.perf_counter()
sorted(db)
t_sort = time.perf_counter() - t0

print(f"100 линейных поисков:  {t_linear_100 * 1000:8.1f} мс")
print(f"100 бинарных поисков:  {t_binary_100 * 1000:8.3f} мс")
print(f"сортировка базы:       {t_sort * 1000:8.1f} мс")

100 линейных поисков:      37.6 мс
100 бинарных поисков:     0.119 мс
сортировка базы:            1.5 мс


In [6]:
# Калькулятор дня: 5 000 поисков = 50 раз по 100.
day_linear = t_linear_100 * DAILY_QUERIES / QUERY_COUNT
day_sort_and_binary = t_sort + t_binary_100 * DAILY_QUERIES / QUERY_COUNT

print(f"за день, линейно:             {day_linear * 1000:9.1f} мс")
print(f"за день, сортировка + бинарный: {day_sort_and_binary * 1000:9.1f} мс")
print(f"выигрыш: {day_linear / day_sort_and_binary:,.0f} раз")

linear_per_query = t_linear_100 / QUERY_COUNT
binary_per_query = t_binary_100 / QUERY_COUNT
saved_per_query = linear_per_query - binary_per_query
break_even_queries = math.floor(t_sort / saved_per_query) + 1 if saved_per_query > 0 else None
print("Минимум запросов для окупаемости в этой модели:", break_even_queries)
assert linear_search([1, 3, 5], 3) == 1
assert binary_search([1, 3, 5], 3) == 1
assert linear_search([], 1) == binary_search([], 1) == -1
assert linear_search([1, 3, 5], 2) == binary_search([1, 3, 5], 2) == -1
assert count_steps(0) == 0 and count_steps(3) == 27

за день, линейно:                1879.5 мс
за день, сортировка + бинарный:       7.4 мс
выигрыш: 253 раз
Минимум запросов для окупаемости в этой модели: 4


### Ответ (задача 3)

Выбираю один раз отсортировать базу и затем искать бинарно: стоимость подготовки O(n log n),
после неё Q запросов требуют O(Q log n) вместо O(Qn) у линейного поиска.
Замер ниже подтверждает выбор для заданной серии запросов.

В сохранённом запуске 100 линейных поисков заняли 37.590 мс,
100 бинарных — 0.119 мс, сортировка — 1.493 мс.
По пересчёту на 5000 запросов получаем 1879.511 мс линейно
против 7.422 мс с подготовкой, то есть выигрыш примерно
в 253.2 раза; прогноз о выигрыше больше 100 раз подтвердился.
Это экстраполяция пакета из 100 запросов, а не прямой замер всех 5000.
Для изменяющейся базы нужно дополнительно учитывать обновление порядка; время зависит от среды и запросов.

**Дополнительно. Точка окупаемости.** Пусть L и B — измеренные средние времена одного линейного
и бинарного запроса, а S — время сортировки. Предварительная сортировка выгодна при S + QB < QL,
то есть Q > S / (L − B), если L > B. Минимальное целое число запросов равно floor(S / (L − B)) + 1.
При L ≤ B положительная стоимость подготовки в этой модели не окупается.
Результат для текущих измерений печатается вместе с оценкой дня; это оценка, а не универсальный порог.

Для текущего запуска модель даёт окупаемость с 4 запросов в день.
При малом числе запросов этот порог особенно чувствителен к шуму измерений и составу запросов.

---

## Задача 4 (со звёздочкой). Миллиард — это сколько шагов?

Сколько шагов в худшем случае у бинарного поиска по массиву
из 10⁹ элементов? Сначала — гипотеза письменно, потом запускайте ячейку.

Гипотеза до запуска: максимум 30 итераций цикла поиска.

In [7]:
def binary_search_steps(haystack, target):
    lo, hi = 0, len(haystack) - 1
    steps = 0
    while lo <= hi:
        mid = (lo + hi) // 2
        steps += 1
        if haystack[mid] == target:
            return steps
        if haystack[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return steps


billion = range(10**9)  # ленивый объект: миллиарда в памяти нет
for x in [0, 123_456_789, 999_999_999]:
    print(f"иском {x:11,d}: {binary_search_steps(billion, x)} шагов")
print(f"log2(10^9) = {math.log2(10**9):.1f}")

assert binary_search_steps(range(0), 1) == 0
assert binary_search_steps(range(1), 0) == 1
assert binary_search_steps(billion, 999_999_999) == 30
assert binary_search_steps(billion, 1_000_000_000) == 30

иском           0: 29 шагов
иском 123,456,789: 30 шагов
иском 999,999,999: 30 шагов
log2(10^9) = 29.9


### Ответ (задача 4)

2²⁹ = 536 870 912 < 10⁹ < 1 073 741 824 = 2³⁰, поэтому требуется не более 30 шагов.
Для данного алгоритма точная верхняя граница при n > 0 — floor(log₂ n) + 1;
при n = 10⁹ она равна 30 и достигается, например, при поиске последнего элемента.
После каждой неудачной итерации остаётся не больше половины кандидатов.
Шаг здесь означает одну итерацию цикла, а не одну отдельную операцию сравнения Python.
Объект range позволяет проверить это без выделения памяти под миллиард чисел.

---

## Сдача

Чек-лист перед отправкой:

- имя и группа вписаны в титул;
- демо-ячейки прогнаны, выводы на месте;
- все ячейки «### Ответ» заполнены;
- файл сохранён и отправлен преподавателю (.ipynb).

Если какая-то ячейка не запускается или что-то пошло не так — напишите
прямо в этой ячейке, что произошло: это нормально.